# Gemma-2 9B LoRA External GPU Training

Train 3 grouped folds of Gemma-2 LoRA adapters. Frozen Gemma weights stay fp16, trainable LoRA/head parameters are fp32, CUDA is required by default, and bitsandbytes is only required when LLM_USE_4BIT=1.


In [ ]:
import os
# Optional: set this only if Kaggle mounts the private dataset at a known path.
# os.environ['LLM_DATA_DIR'] = '/kaggle/input/llm-classification-finetuning-private-data'
os.environ.setdefault('LLM_N_FOLDS', '3')
os.environ.setdefault('LLM_NUM_EPOCHS', '2')
os.environ.setdefault('LLM_MAX_LENGTH', '2048')
os.environ.setdefault('LLM_USE_4BIT', '0')
os.environ.setdefault('LLM_REQUIRE_CUDA', '1')
os.environ.setdefault('LLM_TRAINABLE_PARAM_DTYPE', 'float32')
os.environ.setdefault('LLM_NUM_WORKERS', '0')


In [ ]:
r"""
Experiment: Gemma-2-9B + LoRA/QLoRA + Grouped CV + A/B Swap + Swap TTA
-----------------------------------------------------------------------
Kaggle training script for the LLM Classification Finetuning competition
using a decoder-only model (Gemma-2-9B) with LoRA adapters. The default path
is fp16 LoRA for high-memory external GPUs such as RTX PRO 6000-class cards.
Set LLM_USE_4BIT=1 to switch back to 4-bit NF4 QLoRA for constrained GPUs.

Key differences from the ModernBERT encoder pipeline:
  - LEFT padding (decoder models attend left-to-right)
  - Last non-padding token used for classification (no CLS token)
  - BOS-only input format (no SEP tokens)
  - fp16 LoRA by default; optional 4-bit NF4 QLoRA via BitsAndBytesConfig
  - LoRA targets: q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj
  - Classification head saved via modules_to_save=["score"]
  - Default 3-fold grouped CV, max_length=2048

Recommended full run:
    .\\scripts\\submit_to_kaggle.ps1 `
        -ScriptFile ".\\src\\generate_submission_gemma2_lora.py" `
        -Message "Gemma-2-9B QLoRA grouped CV full run" `
        -EnableGpu `
        -Model "google/gemma-2/transformers/gemma-2-9b/1" `
        -Dataset "your-username/your-offline-wheels-dataset"
"""

from __future__ import annotations

import gc
import importlib.metadata as importlib_metadata
import json
import math
import os
import random
import re
import subprocess
import sys
import time
from dataclasses import dataclass
from pathlib import Path


# ---------------------------------------------------------------------------
# Dependency bootstrap
# ---------------------------------------------------------------------------

os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("HF_DATASETS_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

MIN_DEPENDENCIES = {
    "transformers": "4.52.1",
    "accelerate": "0.29.3",
    "peft": "0.11.1",
}
BNB_MIN_VERSION = "0.43.0"


def parse_version_tuple(version: str) -> tuple[int, ...]:
    parts = re.findall(r"\d+", version)
    return tuple(int(part) for part in parts[:3]) if parts else (0,)


def installed_version(package_name: str) -> str | None:
    try:
        return importlib_metadata.version(package_name)
    except importlib_metadata.PackageNotFoundError:
        return None


def package_is_usable(package_name: str, minimum_version: str) -> bool:
    version = installed_version(package_name)
    if version is None:
        return False
    return parse_version_tuple(version) >= parse_version_tuple(minimum_version)


def offline_wheel_dirs() -> list[str]:
    input_root = Path("/kaggle/input")
    if not input_root.exists():
        return []
    return sorted({str(path.parent) for path in input_root.rglob("*.whl")})


def run_pip_install(command_args: list[str]) -> None:
    print("Running:", " ".join(command_args))
    subprocess.check_call(command_args)


def install_requirement_offline(package_name: str, requirement: str) -> None:
    wheel_dirs = offline_wheel_dirs()
    if not wheel_dirs:
        raise RuntimeError(
            f"{package_name} is missing or too old, and no offline wheel files were found under "
            "/kaggle/input. Attach a Kaggle Dataset containing compatible wheels for "
            "peft, accelerate, transformers, and any optional quantisation dependencies."
        )

    base_command = [
        sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
    ]

    offline_command = base_command + ["--no-index"]
    for wheel_dir in wheel_dirs:
        offline_command.extend(["--find-links", wheel_dir])
    offline_command.append(requirement)

    print(f"Installing {package_name} from attached offline wheels...")
    try:
        run_pip_install(offline_command)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            f"Offline wheel install failed for {package_name}. Make sure the attached wheel "
            f"dataset includes {requirement} plus all required dependencies. Original error: {exc}"
        ) from exc


def ensure_dependencies() -> None:
    for package_name, minimum_version in MIN_DEPENDENCIES.items():
        version = installed_version(package_name)
        if package_is_usable(package_name, minimum_version):
            print(f"{package_name} {version} is available.")
            continue

        requirement = f"{package_name}>={minimum_version}"
        if package_name == "transformers":
            requirement = f"{package_name}>={minimum_version},<5.0.0"

        if version is None:
            print(f"{package_name} is not installed; installing {requirement} from offline wheels.")
        else:
            print(f"{package_name} {version} is too old; installing {requirement} from offline wheels.")
        install_requirement_offline(package_name, requirement)


def use_4bit_requested() -> bool:
    raw = os.environ.get("LLM_USE_4BIT")
    if raw is None:
        return False
    return raw.strip().lower() in {"1", "true", "yes", "y", "on"}


def ensure_bitsandbytes_if_needed() -> None:
    if not use_4bit_requested():
        print("LLM_USE_4BIT is disabled; skipping bitsandbytes dependency check.")
        return

    if package_is_usable("bitsandbytes", BNB_MIN_VERSION):
        print(f"bitsandbytes {installed_version('bitsandbytes')} is available.")
        return

    requirement = f"bitsandbytes>={BNB_MIN_VERSION}"
    version = installed_version("bitsandbytes")
    if version is None:
        print(f"bitsandbytes is not installed; installing {requirement} from offline wheels.")
    else:
        print(f"bitsandbytes {version} is too old; installing {requirement} from offline wheels.")
    install_requirement_offline("bitsandbytes", requirement)


ensure_dependencies()
ensure_bitsandbytes_if_needed()


import numpy as np
import pandas as pd
import torch
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    get_peft_model_state_dict,
    prepare_model_for_kbit_training,
    set_peft_model_state_dict,
)
from sklearn.metrics import log_loss
from sklearn.model_selection import GroupKFold
from torch import nn
from torch.utils.data import DataLoader
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup,
)


# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

OFFLINE_MODEL_PATHS = [
    "/kaggle/input/gemma-2/transformers/gemma-2-9b/1",
    "/kaggle/input/gemma-2/Transformers/gemma-2-9b/1",
    "/kaggle/input/google-gemma-2-9b",
    "/kaggle/input/gemma-2-9b",
    "/kaggle/input/gemma2-9b",
]

COMPETITION_INPUT_DIR = Path("/kaggle/input/llm-classification-finetuning")
PRIVATE_DATASET_INPUT_DIRS = [
    Path("/kaggle/input/llm-classification-finetuning-private-data"),
    Path("/kaggle/input/llm-classification-finetuning-private"),
    Path("/kaggle/input/llm-classification-private-data"),
]
PRIVATE_DATASET_HINTS = ("llm-classification", "finetuning", "private")
EXCLUDED_DATASET_HINTS = ("nvidia", "nemotron", "reasoning-challenge")
LOCAL_INPUT_DIR = Path.cwd() / "data" / "raw"
DEFAULT_OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()

LABEL_COLUMNS = ["winner_model_a", "winner_model_b", "winner_tie"]
NUM_LABELS = len(LABEL_COLUMNS)
SWAP_LABEL_MAP = np.array([1, 0, 2], dtype=np.int64)


def env_int(name: str, default: int) -> int:
    return int(os.environ.get(name, str(default)))


def env_float(name: str, default: float) -> float:
    return float(os.environ.get(name, str(default)))


def env_bool(name: str, default: bool) -> bool:
    raw = os.environ.get(name)
    if raw is None:
        return default
    return raw.strip().lower() in {"1", "true", "yes", "y", "on"}


# Decoder-specific defaults for high-memory external GPU training.
N_FOLDS = env_int("LLM_N_FOLDS", 3)
RANDOM_STATE = env_int("LLM_RANDOM_STATE", 42)
MAX_LENGTH_REQUESTED = env_int("LLM_MAX_LENGTH", 2048)
NUM_EPOCHS = env_int("LLM_NUM_EPOCHS", 2)
TRAIN_BATCH_SIZE = env_int("LLM_TRAIN_BATCH_SIZE", 1)
INFER_BATCH_SIZE = env_int("LLM_INFER_BATCH_SIZE", 2)
GRADIENT_ACCUMULATION_STEPS = env_int("LLM_GRADIENT_ACCUMULATION_STEPS", 16)
LEARNING_RATE = env_float("LLM_LEARNING_RATE", 1e-4)
WEIGHT_DECAY = env_float("LLM_WEIGHT_DECAY", 0.01)
WARMUP_RATIO = env_float("LLM_WARMUP_RATIO", 0.08)
LABEL_SMOOTHING = env_float("LLM_LABEL_SMOOTHING", 0.02)
MAX_GRAD_NORM = env_float("LLM_MAX_GRAD_NORM", 1.0)
TRAIN_ROW_LIMIT = env_int("LLM_TRAIN_ROW_LIMIT", 0)
NUM_WORKERS = env_int("LLM_NUM_WORKERS", 0)
USE_FP16 = env_bool("LLM_USE_FP16", True)
USE_GRADIENT_CHECKPOINTING = env_bool("LLM_USE_GRADIENT_CHECKPOINTING", True)
REQUIRE_CUDA = env_bool("LLM_REQUIRE_CUDA", True)
TRAINABLE_PARAM_DTYPE = os.environ.get("LLM_TRAINABLE_PARAM_DTYPE", "float32").strip().lower()

# LoRA — decoder targets for Gemma-2
LORA_RANK = env_int("LLM_LORA_RANK", 16)
LORA_ALPHA = env_int("LLM_LORA_ALPHA", 32)
LORA_DROPOUT = env_float("LLM_LORA_DROPOUT", 0.1)
LORA_TARGET_MODULES = [
    item.strip()
    for item in os.environ.get(
        "LLM_LORA_TARGET_MODULES",
        "q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj",
    ).split(",")
    if item.strip()
]
MODULES_TO_SAVE = [
    item.strip()
    for item in os.environ.get("LLM_MODULES_TO_SAVE", "score").split(",")
    if item.strip()
]

# 4-bit NF4 quantisation. Keep disabled by default for RTX PRO 6000-class
# training, where fp16 LoRA should usually preserve more signal.
USE_4BIT = env_bool("LLM_USE_4BIT", False)
BNB_QUANT_TYPE = os.environ.get("LLM_BNB_QUANT_TYPE", "nf4").strip()
BNB_COMPUTE_DTYPE = os.environ.get("LLM_BNB_COMPUTE_DTYPE", "float16").strip()
BNB_DOUBLE_QUANT = env_bool("LLM_BNB_DOUBLE_QUANT", True)

# Token budget shares (same as ModernBERT: 18% prompt, 41% each response)
PROMPT_SHARE = env_float("LLM_PROMPT_SHARE", 0.18)
RESPONSE_A_SHARE = env_float("LLM_RESPONSE_A_SHARE", 0.41)
RESPONSE_B_SHARE = env_float("LLM_RESPONSE_B_SHARE", 0.41)
PROMPT_HEAD_RATIO = env_float("LLM_PROMPT_HEAD_RATIO", 0.80)
RESPONSE_HEAD_RATIO = env_float("LLM_RESPONSE_HEAD_RATIO", 0.72)
PROBABILITY_EPS = env_float("LLM_PROBABILITY_EPS", 1e-7)

OUTPUT_DIR = Path(os.environ.get("LLM_OUTPUT_DIR", str(DEFAULT_OUTPUT_DIR)))


@dataclass(frozen=True)
class EncodedRow:
    prompt_ids: list[int]
    response_a_ids: list[int]
    response_b_ids: list[int]


# ---------------------------------------------------------------------------
# General helpers
# ---------------------------------------------------------------------------


def print_rule(title: str) -> None:
    print()
    print("=" * 90)
    print(title)
    print("=" * 90)


def format_seconds(seconds: float) -> str:
    minutes, sec = divmod(int(seconds), 60)
    hours, minutes = divmod(minutes, 60)
    return f"{hours}h {minutes}m {sec}s"


def resolve_model_name() -> str:
    env_model_path = os.environ.get("LLM_MODEL_PATH")
    if env_model_path:
        path = Path(env_model_path)
        if path.exists():
            print(f"Using offline Gemma-2 weights from LLM_MODEL_PATH: {path}")
            return str(path)
        raise FileNotFoundError(f"LLM_MODEL_PATH was set but does not exist: {path}")

    for path in OFFLINE_MODEL_PATHS:
        if Path(path).exists():
            print(f"Found offline Gemma-2 weights at: {path}")
            return path

    if Path("/kaggle/input").exists():
        for config_path in sorted(Path("/kaggle/input").rglob("config.json")):
            try:
                payload = json.loads(config_path.read_text(encoding="utf-8"))
            except Exception:
                continue
            model_type = str(payload.get("model_type", "")).lower()
            architectures = " ".join(str(a) for a in payload.get("architectures", [])).lower()
            if "gemma2" in model_type or "gemma" in architectures:
                print(f"Discovered offline Gemma-2 weights at: {config_path.parent}")
                return str(config_path.parent)

    raise FileNotFoundError(
        "No offline Gemma-2 weights were found under /kaggle/input. Attach the Kaggle "
        "Gemma-2 model/dataset, or set LLM_MODEL_PATH to the attached model directory."
    )


MODEL_NAME = resolve_model_name()


def seed_everything(seed: int = RANDOM_STATE) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


def seed_worker(worker_id: int) -> None:
    worker_seed = (RANDOM_STATE + worker_id) % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def find_data_dir() -> Path:
    env_data_dir = os.environ.get("LLM_DATA_DIR")
    if env_data_dir:
        path = Path(env_data_dir)
        if (path / "train.csv").exists() and (path / "test.csv").exists():
            print(f"Using data directory from LLM_DATA_DIR: {path}")
            return path
        print(f"WARNING: LLM_DATA_DIR does not contain train.csv and test.csv, continuing search: {path}")

    for path in PRIVATE_DATASET_INPUT_DIRS:
        if (path / "train.csv").exists() and (path / "test.csv").exists():
            print(f"Using attached private competition data: {path}")
            return path

    if COMPETITION_INPUT_DIR.exists():
        return COMPETITION_INPUT_DIR

    if Path("/kaggle/input").exists():
        candidates = []
        for train_path in sorted(Path("/kaggle/input").rglob("train.csv")):
            parent = train_path.parent
            if not (parent / "test.csv").exists():
                continue
            lowered = str(parent).lower()
            if any(hint in lowered for hint in EXCLUDED_DATASET_HINTS):
                continue
            score = sum(hint in lowered for hint in PRIVATE_DATASET_HINTS)
            candidates.append((score, str(parent), parent))
        if candidates:
            candidates.sort(key=lambda item: (-item[0], item[1]))
            selected = candidates[0][2]
            print(f"Discovered competition data directory: {selected}")
            return selected

    if (LOCAL_INPUT_DIR / "train.csv").exists() and (LOCAL_INPUT_DIR / "test.csv").exists():
        return LOCAL_INPUT_DIR

    candidates = []
    if Path("/kaggle/input").exists():
        for train_path in sorted(Path("/kaggle/input").rglob("train.csv"))[:50]:
            candidates.append(str(train_path.parent))
    candidate_text = "\n".join(f"- {candidate}" for candidate in candidates) or "- no train.csv candidates found"
    raise FileNotFoundError(
        "Could not find LLM Classification train.csv and test.csv. Attach the private LLM "
        "classification dataset, or set LLM_DATA_DIR to the actual mounted dataset folder.\n"
        f"Candidate train.csv folders seen under /kaggle/input:\n{candidate_text}"
    )


def load_model_config():
    config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
    # Disable torch.compile for compatibility
    if hasattr(config, "reference_compile"):
        config.reference_compile = False
    return config


def effective_max_length(config) -> int:
    model_max_length = getattr(config, "max_position_embeddings", None)
    if model_max_length and MAX_LENGTH_REQUESTED > model_max_length:
        print(
            f"Requested max length {MAX_LENGTH_REQUESTED} exceeds model limit "
            f"{model_max_length}; using {model_max_length}."
        )
        return int(model_max_length)
    return MAX_LENGTH_REQUESTED


def validate_input_frames(train_df: pd.DataFrame, test_df: pd.DataFrame) -> None:
    train_required = ["id", "prompt", "response_a", "response_b", *LABEL_COLUMNS]
    test_required = ["id", "prompt", "response_a", "response_b"]
    missing_train = [col for col in train_required if col not in train_df.columns]
    missing_test = [col for col in test_required if col not in test_df.columns]
    if missing_train:
        raise ValueError(f"train.csv is missing required columns: {missing_train}")
    if missing_test:
        raise ValueError(f"test.csv is missing required columns: {missing_test}")

    label_matrix = train_df[LABEL_COLUMNS].astype(float).values
    row_sums = label_matrix.sum(axis=1)
    invalid_count = int((row_sums != 1).sum())
    if invalid_count:
        raise ValueError(f"Found {invalid_count} train rows without exactly one winning label.")


def build_labels(df: pd.DataFrame) -> np.ndarray:
    return df[LABEL_COLUMNS].astype(float).values.argmax(axis=1).astype(np.int64)


def build_groups(df: pd.DataFrame) -> np.ndarray:
    group_ids, _ = pd.factorize(df["prompt"].fillna("").astype(str), sort=False)
    return group_ids.astype(np.int64)


def limit_training_rows(df: pd.DataFrame) -> pd.DataFrame:
    if TRAIN_ROW_LIMIT <= 0 or TRAIN_ROW_LIMIT >= len(df):
        return df
    label_ids = build_labels(df)
    limited_indices = (
        pd.DataFrame({"row_index": np.arange(len(df)), "label": label_ids})
        .groupby("label", group_keys=False)
        .sample(frac=TRAIN_ROW_LIMIT / len(df), random_state=RANDOM_STATE)
        .head(TRAIN_ROW_LIMIT)["row_index"]
        .to_numpy()
    )
    if len(limited_indices) < TRAIN_ROW_LIMIT:
        remaining = TRAIN_ROW_LIMIT - len(limited_indices)
        pool = np.setdiff1d(np.arange(len(df)), limited_indices, assume_unique=False)
        extra = np.random.default_rng(RANDOM_STATE).choice(pool, size=remaining, replace=False)
        limited_indices = np.concatenate([limited_indices, extra])
    limited_df = df.iloc[limited_indices].sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)
    if len(limited_df) < N_FOLDS * NUM_LABELS:
        raise ValueError(f"LLM_TRAIN_ROW_LIMIT={TRAIN_ROW_LIMIT} is too small for {N_FOLDS} folds.")
    print(f"Using a limited training set of {len(limited_df):,} rows.")
    return limited_df


def normalize_probabilities(probs: np.ndarray) -> np.ndarray:
    probs = np.asarray(probs, dtype=np.float64)
    probs = np.clip(probs, PROBABILITY_EPS, 1.0 - PROBABILITY_EPS)
    row_sums = probs.sum(axis=1, keepdims=True)
    return (probs / row_sums).astype(np.float32)


def validate_submission(submission_df: pd.DataFrame, expected_rows: int) -> None:
    expected_columns = ["id", *LABEL_COLUMNS]
    if list(submission_df.columns) != expected_columns:
        raise ValueError(f"Submission columns must be {expected_columns}; got {list(submission_df.columns)}")
    if len(submission_df) != expected_rows:
        raise ValueError(f"Submission row count mismatch: {len(submission_df)} != {expected_rows}")
    probs = submission_df[LABEL_COLUMNS].values
    if not np.isfinite(probs).all():
        raise ValueError("Submission probabilities contain NaN or infinite values.")
    if (probs < 0).any() or (probs > 1).any():
        raise ValueError("Submission probabilities must stay within [0, 1].")
    max_sum_error = float(np.abs(probs.sum(axis=1) - 1.0).max())
    if max_sum_error > 1e-5:
        raise ValueError(f"Submission probability rows do not sum to 1. Max error: {max_sum_error}")


# ---------------------------------------------------------------------------
# Tokenization and truncation (decoder-specific: BOS only, no CLS/SEP)
# ---------------------------------------------------------------------------


def truncate_head_tail(token_ids: list[int], budget: int, head_ratio: float) -> list[int]:
    if budget <= 0:
        return []
    if len(token_ids) <= budget:
        return token_ids
    head_count = max(1, int(round(budget * head_ratio)))
    head_count = min(head_count, budget - 1)
    tail_count = budget - head_count
    if tail_count <= 0:
        return token_ids[:budget]
    return token_ids[:head_count] + token_ids[-tail_count:]


def allocate_budgets(lengths: list[int], total_budget: int, shares: list[float]) -> list[int]:
    budgets = [min(length, int(total_budget * share)) for length, share in zip(lengths, shares)]
    assigned = sum(budgets)
    remaining = max(0, total_budget - assigned)
    while remaining > 0:
        candidate = max(range(len(lengths)), key=lambda idx: (lengths[idx] - budgets[idx], lengths[idx]))
        if budgets[candidate] >= lengths[candidate]:
            break
        budgets[candidate] += 1
        remaining -= 1
    return budgets


def build_model_inputs(
    encoded_row: EncodedRow,
    tokenizer,
    max_length: int,
    swap_responses: bool,
) -> tuple[list[int], list[int]]:
    """Build decoder-format input: [BOS] prompt \\n\\n response_a \\n\\n response_b."""
    prompt_ids = encoded_row.prompt_ids
    response_a_ids = encoded_row.response_b_ids if swap_responses else encoded_row.response_a_ids
    response_b_ids = encoded_row.response_a_ids if swap_responses else encoded_row.response_b_ids

    # Reserve 1 token for BOS (decoder models have no CLS/SEP)
    content_budget = max_length - 1
    budgets = allocate_budgets(
        lengths=[len(prompt_ids), len(response_a_ids), len(response_b_ids)],
        total_budget=content_budget,
        shares=[PROMPT_SHARE, RESPONSE_A_SHARE, RESPONSE_B_SHARE],
    )

    prompt_final = truncate_head_tail(prompt_ids, budgets[0], PROMPT_HEAD_RATIO)
    response_a_final = truncate_head_tail(response_a_ids, budgets[1], RESPONSE_HEAD_RATIO)
    response_b_final = truncate_head_tail(response_b_ids, budgets[2], RESPONSE_HEAD_RATIO)

    bos_id = tokenizer.bos_token_id if tokenizer.bos_token_id is not None else tokenizer.cls_token_id
    input_ids = [bos_id, *prompt_final, *response_a_final, *response_b_final]
    return input_ids, [1] * len(input_ids)


def encode_texts(tokenizer, texts: list[str], batch_size: int = 256) -> list[list[int]]:
    all_ids: list[list[int]] = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start : start + batch_size]
        tokenized = tokenizer(batch, add_special_tokens=False, truncation=False, verbose=False)
        all_ids.extend(tokenized["input_ids"])
    return all_ids


def pretokenize_dataframe(df: pd.DataFrame, tokenizer) -> list[EncodedRow]:
    prompt_ids = encode_texts(
        tokenizer,
        (
            "<task>\n"
            "Judge which assistant response better answers the user prompt. "
            "Return one of: A wins, B wins, or tie.\n"
            "</task>\n\n"
            "<prompt>\n"
            + df["prompt"].fillna("").astype(str)
            + "\n</prompt>\n\n"
        ).tolist(),
    )
    response_a_ids = encode_texts(
        tokenizer,
        ("<response_a>\n" + df["response_a"].fillna("").astype(str) + "\n</response_a>\n\n").tolist(),
    )
    response_b_ids = encode_texts(
        tokenizer,
        ("<response_b>\n" + df["response_b"].fillna("").astype(str) + "\n</response_b>\n\n<verdict>").tolist(),
    )
    return [EncodedRow(p, a, b) for p, a, b in zip(prompt_ids, response_a_ids, response_b_ids)]


# ---------------------------------------------------------------------------
# Dataset (LEFT padding for decoder models)
# ---------------------------------------------------------------------------


class PreferenceDataset:
    def __init__(
        self,
        encoded_rows: list[EncodedRow],
        row_indices: np.ndarray,
        labels: np.ndarray | None,
        tokenizer,
        max_length: int,
        swap_flags: np.ndarray | None = None,
        swap_labels: bool = False,
    ) -> None:
        self.encoded_rows = encoded_rows
        self.row_indices = row_indices.astype(np.int64)
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.swap_flags = (
            swap_flags.astype(bool) if swap_flags is not None else np.zeros(len(self.row_indices), dtype=bool)
        )
        self.swap_labels = swap_labels

    def __len__(self) -> int:
        return len(self.row_indices)

    def __getitem__(self, index: int) -> dict[str, list[int] | int]:
        row_index = int(self.row_indices[index])
        swap = bool(self.swap_flags[index])
        input_ids, attention_mask = build_model_inputs(
            self.encoded_rows[row_index],
            self.tokenizer,
            self.max_length,
            swap,
        )
        item: dict[str, list[int] | int] = {"input_ids": input_ids, "attention_mask": attention_mask}
        if self.labels is not None:
            label = int(self.labels[row_index])
            if self.swap_labels and swap:
                label = int(SWAP_LABEL_MAP[label])
            item["labels"] = label
        return item


def build_collate_fn(tokenizer):
    """LEFT-padding collate for decoder models."""
    pad_id = tokenizer.pad_token_id

    def collate_fn(batch):
        max_len = max(len(item["input_ids"]) for item in batch)
        input_ids, attention_mask, labels = [], [], []
        for item in batch:
            pad_width = max_len - len(item["input_ids"])
            # LEFT padding: pad tokens at the start
            input_ids.append([pad_id] * pad_width + item["input_ids"])
            attention_mask.append([0] * pad_width + item["attention_mask"])
            if "labels" in item:
                labels.append(item["labels"])
        payload = {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        }
        if labels:
            payload["labels"] = torch.tensor(labels, dtype=torch.long)
        return payload

    return collate_fn


def build_train_indices_with_swap(indices: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    original_indices = indices.astype(np.int64)
    swapped_indices = indices.astype(np.int64)
    merged_indices = np.concatenate([original_indices, swapped_indices])
    swap_flags = np.concatenate(
        [
            np.zeros(len(original_indices), dtype=bool),
            np.ones(len(swapped_indices), dtype=bool),
        ]
    )
    return merged_indices, swap_flags


# ---------------------------------------------------------------------------
# Modeling, training, and inference
# ---------------------------------------------------------------------------


def build_quantization_config():
    """Create BitsAndBytesConfig for 4-bit NF4 quantisation."""
    if not USE_4BIT:
        return None
    compute_dtype_map = {
        "float16": torch.float16,
        "bfloat16": torch.bfloat16,
        "float32": torch.float32,
    }
    compute_dtype = compute_dtype_map.get(BNB_COMPUTE_DTYPE, torch.float16)
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type=BNB_QUANT_TYPE,
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=BNB_DOUBLE_QUANT,
    )


def torch_dtype_from_name(dtype_name: str):
    dtype_map = {
        "float16": torch.float16,
        "fp16": torch.float16,
        "bfloat16": torch.bfloat16,
        "bf16": torch.bfloat16,
        "float32": torch.float32,
        "fp32": torch.float32,
    }
    return dtype_map.get(dtype_name.strip().lower(), torch.float16)


def cast_trainable_parameters(model, dtype: torch.dtype) -> None:
    converted = 0
    for param in model.parameters():
        if param.requires_grad and param.dtype != dtype:
            param.data = param.data.to(dtype)
            converted += param.numel()
    if converted:
        print(f"Cast {converted:,} trainable parameters to {dtype}.")


def trainable_parameter_dtype_counts(model) -> dict[str, int]:
    counts: dict[str, int] = {}
    for param in model.parameters():
        if param.requires_grad:
            key = str(param.dtype).replace("torch.", "")
            counts[key] = counts.get(key, 0) + param.numel()
    return counts


def has_fp16_trainable_parameters(model) -> bool:
    return any(param.requires_grad and param.dtype == torch.float16 for param in model.parameters())


def build_model_with_lora(config, tokenizer):
    """Load Gemma-2 and apply LoRA adapters."""
    bnb_config = build_quantization_config()

    model_dtype = torch_dtype_from_name(os.environ.get("LLM_MODEL_DTYPE", "float16"))
    load_kwargs = {"config": config, "torch_dtype": model_dtype}
    if bnb_config is not None:
        load_kwargs["quantization_config"] = bnb_config
        load_kwargs["device_map"] = "auto"

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, **load_kwargs)

    # Ensure pad_token_id is set on the model config
    if model.config.pad_token_id is None:
        model.config.pad_token_id = tokenizer.pad_token_id

    # Prepare for k-bit training (freeze quantised layers, cast norms to fp32)
    if bnb_config is not None:
        model = prepare_model_for_kbit_training(
            model, use_gradient_checkpointing=USE_GRADIENT_CHECKPOINTING
        )

    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGET_MODULES,
        modules_to_save=MODULES_TO_SAVE,
        bias="none",
    )
    model = get_peft_model(model, lora_config)

    trainable_dtype = torch_dtype_from_name(TRAINABLE_PARAM_DTYPE)
    if trainable_dtype == torch.float16:
        print(
            "WARNING: LLM_TRAINABLE_PARAM_DTYPE=float16 can break GradScaler; "
            "float32 is recommended for LoRA/head parameters."
        )
    else:
        cast_trainable_parameters(model, trainable_dtype)

    if USE_GRADIENT_CHECKPOINTING and hasattr(model, "gradient_checkpointing_enable"):
        try:
            model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        except TypeError:
            model.gradient_checkpointing_enable()
        if hasattr(model, "enable_input_require_grads"):
            model.enable_input_require_grads()

    if hasattr(model, "print_trainable_parameters"):
        model.print_trainable_parameters()
    print(f"Trainable parameter dtypes: {trainable_parameter_dtype_counts(model)}")

    return model


def build_optimizer(model):
    trainable_params = [param for param in model.parameters() if param.requires_grad]
    if not trainable_params:
        raise RuntimeError("No trainable parameters found after applying LoRA.")
    return torch.optim.AdamW(trainable_params, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)


@np.errstate(over="ignore")
def softmax_np(logits: np.ndarray) -> np.ndarray:
    shifted = logits - logits.max(axis=1, keepdims=True)
    exp_scores = np.exp(shifted)
    return normalize_probabilities(exp_scores / exp_scores.sum(axis=1, keepdims=True))


def autocast_context(device):
    if hasattr(torch, "amp") and hasattr(torch.amp, "autocast"):
        return torch.amp.autocast("cuda", enabled=USE_FP16 and device.type == "cuda")
    return torch.cuda.amp.autocast(enabled=USE_FP16 and device.type == "cuda")


def predict_probabilities(model, loader, device) -> np.ndarray:
    model.eval()
    all_logits = []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            with autocast_context(device):
                logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            all_logits.append(logits.detach().cpu().to(torch.float32).numpy())
    return softmax_np(np.vstack(all_logits))


def predict_with_swap_tta(model, loader_original, loader_swapped, device) -> np.ndarray:
    original_probs = predict_probabilities(model, loader_original, device)
    swapped_probs = predict_probabilities(model, loader_swapped, device)
    swapped_back = swapped_probs[:, SWAP_LABEL_MAP]
    return normalize_probabilities(0.5 * (original_probs + swapped_back))


def train_one_epoch(model, loader, optimizer, scheduler, criterion, scaler, device, epoch_idx: int) -> float:
    model.train()
    optimizer.zero_grad(set_to_none=True)
    running_loss = 0.0

    for step, batch in enumerate(loader, start=1):
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        targets = batch["labels"].to(device, non_blocking=True)

        with autocast_context(device):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs.logits, targets)
            scaled_loss = loss / GRADIENT_ACCUMULATION_STEPS

        scaler.scale(scaled_loss).backward()
        running_loss += float(loss.detach().cpu())

        if step % GRADIENT_ACCUMULATION_STEPS == 0 or step == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            old_scale = scaler.get_scale() if scaler.is_enabled() else None
            scaler.step(optimizer)
            scaler.update()
            new_scale = scaler.get_scale() if scaler.is_enabled() else None
            optimizer.zero_grad(set_to_none=True)
            if old_scale is None or new_scale >= old_scale:
                scheduler.step()

        if step % 200 == 0 or step == len(loader):
            avg_loss = running_loss / step
            print(f"Epoch {epoch_idx} | step {step:,}/{len(loader):,} | avg_loss={avg_loss:.5f}")

    return running_loss / max(1, len(loader))


def build_grad_scaler(model, device):
    enabled = USE_FP16 and device.type == "cuda"
    if enabled and has_fp16_trainable_parameters(model):
        print(
            "WARNING: FP16 trainable parameters detected; disabling GradScaler to avoid "
            "'Attempting to unscale FP16 gradients'. Prefer LLM_TRAINABLE_PARAM_DTYPE=float32."
        )
        enabled = False

    if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
        return torch.amp.GradScaler("cuda", enabled=enabled)
    return torch.cuda.amp.GradScaler(enabled=enabled)


def save_best_adapter(model, tokenizer, save_dir: Path, metrics: dict) -> None:
    save_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)
    (save_dir / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")


def clear_torch_memory() -> None:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------


def main() -> int:
    run_start = time.time()
    seed_everything()
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type != "cuda":
        message = "CUDA is not available. Gemma-2-9B LoRA training needs a GPU accelerator."
        if REQUIRE_CUDA:
            raise RuntimeError(message)
        print(f"WARNING: {message}")

    data_dir = find_data_dir()
    train_df = pd.read_csv(data_dir / "train.csv")
    test_df = pd.read_csv(data_dir / "test.csv")
    validate_input_frames(train_df, test_df)
    train_df = limit_training_rows(train_df)

    config = load_model_config()
    max_length = effective_max_length(config)

    print_rule("Run configuration")
    print(f"Data dir:                      {data_dir}")
    print(f"Output dir:                    {OUTPUT_DIR}")
    print(f"Model:                         {MODEL_NAME}")
    print(f"Device:                        {device}")
    print(f"Train rows:                    {len(train_df):,}")
    print(f"Test rows:                     {len(test_df):,}")
    print(f"Folds:                         {N_FOLDS}")
    print(f"Epochs per fold:               {NUM_EPOCHS}")
    print(f"Max length requested/effective: {MAX_LENGTH_REQUESTED}/{max_length}")
    print(f"Train batch size:              {TRAIN_BATCH_SIZE}")
    print(f"Inference batch size:          {INFER_BATCH_SIZE}")
    print(f"Gradient accumulation:         {GRADIENT_ACCUMULATION_STEPS}")
    print(f"Learning rate:                 {LEARNING_RATE}")
    print(f"LoRA rank/alpha/dropout:       {LORA_RANK}/{LORA_ALPHA}/{LORA_DROPOUT}")
    print(f"LoRA target modules:           {LORA_TARGET_MODULES}")
    print(f"Modules to save:               {MODULES_TO_SAVE}")
    print(f"4-bit quantisation:            {USE_4BIT}")
    print(f"BnB quant type:                {BNB_QUANT_TYPE}")
    print(f"BnB compute dtype:             {BNB_COMPUTE_DTYPE}")
    print(f"FP16 autocast:                 {USE_FP16}")
    print(f"Trainable param dtype:         {TRAINABLE_PARAM_DTYPE}")
    print(f"Gradient checkpointing:        {USE_GRADIENT_CHECKPOINTING}")

    # Decoder tokenizer: LEFT padding, use eos_token as pad_token
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id

    print_rule("Pretokenizing train/test fields")
    encoded_train = pretokenize_dataframe(train_df, tokenizer)
    encoded_test = pretokenize_dataframe(test_df, tokenizer)

    labels = build_labels(train_df)
    groups = build_groups(train_df)
    unique_groups = np.unique(groups)
    if len(unique_groups) < N_FOLDS:
        raise ValueError(f"Need at least {N_FOLDS} unique prompt groups; found {len(unique_groups)}.")

    collate_fn = build_collate_fn(tokenizer)
    splitter = GroupKFold(n_splits=N_FOLDS)
    torch_generator = torch.Generator()
    torch_generator.manual_seed(RANDOM_STATE)

    oof_probs = np.zeros((len(train_df), NUM_LABELS), dtype=np.float32)
    test_probs_accum = np.zeros((len(test_df), NUM_LABELS), dtype=np.float32)
    fold_metrics: list[dict] = []

    for fold_idx, (train_idx, val_idx) in enumerate(splitter.split(train_df, labels, groups=groups), start=1):
        fold_start = time.time()
        print_rule(f"Fold {fold_idx}/{N_FOLDS}")
        print(f"Train rows before swap: {len(train_idx):,}")
        print(f"Validation rows:        {len(val_idx):,}")

        train_indices_aug, train_swap_flags = build_train_indices_with_swap(train_idx)
        val_indices = val_idx.astype(np.int64)
        test_indices = np.arange(len(test_df), dtype=np.int64)

        train_dataset = PreferenceDataset(
            encoded_rows=encoded_train,
            row_indices=train_indices_aug,
            labels=labels,
            tokenizer=tokenizer,
            max_length=max_length,
            swap_flags=train_swap_flags,
            swap_labels=True,
        )
        val_dataset_orig = PreferenceDataset(
            encoded_rows=encoded_train,
            row_indices=val_indices,
            labels=labels,
            tokenizer=tokenizer,
            max_length=max_length,
        )
        val_dataset_swap = PreferenceDataset(
            encoded_rows=encoded_train,
            row_indices=val_indices,
            labels=labels,
            tokenizer=tokenizer,
            max_length=max_length,
            swap_flags=np.ones(len(val_indices), dtype=bool),
        )
        test_dataset_orig = PreferenceDataset(
            encoded_rows=encoded_test,
            row_indices=test_indices,
            labels=None,
            tokenizer=tokenizer,
            max_length=max_length,
        )
        test_dataset_swap = PreferenceDataset(
            encoded_rows=encoded_test,
            row_indices=test_indices,
            labels=None,
            tokenizer=tokenizer,
            max_length=max_length,
            swap_flags=np.ones(len(test_indices), dtype=bool),
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=TRAIN_BATCH_SIZE,
            shuffle=True,
            collate_fn=collate_fn,
            num_workers=NUM_WORKERS,
            pin_memory=(device.type == "cuda"),
            worker_init_fn=seed_worker,
            generator=torch_generator,
        )
        val_loader_orig = DataLoader(
            val_dataset_orig,
            batch_size=INFER_BATCH_SIZE,
            shuffle=False,
            collate_fn=collate_fn,
            num_workers=NUM_WORKERS,
            pin_memory=(device.type == "cuda"),
        )
        val_loader_swap = DataLoader(
            val_dataset_swap,
            batch_size=INFER_BATCH_SIZE,
            shuffle=False,
            collate_fn=collate_fn,
            num_workers=NUM_WORKERS,
            pin_memory=(device.type == "cuda"),
        )
        test_loader_orig = DataLoader(
            test_dataset_orig,
            batch_size=INFER_BATCH_SIZE,
            shuffle=False,
            collate_fn=collate_fn,
            num_workers=NUM_WORKERS,
            pin_memory=(device.type == "cuda"),
        )
        test_loader_swap = DataLoader(
            test_dataset_swap,
            batch_size=INFER_BATCH_SIZE,
            shuffle=False,
            collate_fn=collate_fn,
            num_workers=NUM_WORKERS,
            pin_memory=(device.type == "cuda"),
        )

        # Quantised model uses device_map="auto"; fp16/bf16 LoRA needs manual placement.
        model = build_model_with_lora(config, tokenizer)
        if not USE_4BIT:
            model.to(device)
        optimizer = build_optimizer(model)
        update_steps_per_epoch = math.ceil(len(train_loader) / GRADIENT_ACCUMULATION_STEPS)
        total_update_steps = max(1, update_steps_per_epoch * NUM_EPOCHS)
        warmup_steps = int(total_update_steps * WARMUP_RATIO)
        scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_update_steps)
        criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
        scaler = build_grad_scaler(model, device)

        best_val_loss = float("inf")
        best_adapter_state = None
        best_epoch = 0
        epoch_history = []

        for epoch_idx in range(1, NUM_EPOCHS + 1):
            epoch_start = time.time()
            train_loss = train_one_epoch(
                model=model,
                loader=train_loader,
                optimizer=optimizer,
                scheduler=scheduler,
                criterion=criterion,
                scaler=scaler,
                device=device,
                epoch_idx=epoch_idx,
            )

            print("Evaluating validation fold with swap TTA...")
            val_probs_tta = predict_with_swap_tta(model, val_loader_orig, val_loader_swap, device)
            val_loss = log_loss(labels[val_idx], val_probs_tta, labels=[0, 1, 2])
            print(
                f"Fold {fold_idx} | epoch {epoch_idx} "
                f"| train_loss={train_loss:.5f} "
                f"| val_log_loss={val_loss:.5f} "
                f"| epoch_runtime={format_seconds(time.time() - epoch_start)}"
            )
            epoch_history.append(
                {
                    "fold": fold_idx,
                    "epoch": epoch_idx,
                    "train_loss": float(train_loss),
                    "val_log_loss": float(val_loss),
                    "runtime_seconds": int(time.time() - epoch_start),
                }
            )

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_epoch = epoch_idx
                best_adapter_state = {
                    key: value.detach().cpu().clone()
                    for key, value in get_peft_model_state_dict(model).items()
                }
                oof_probs[val_idx] = val_probs_tta
                save_best_adapter(
                    model=model,
                    tokenizer=tokenizer,
                    save_dir=OUTPUT_DIR / f"fold_{fold_idx}",
                    metrics={
                        "fold": fold_idx,
                        "best_epoch": best_epoch,
                        "best_val_log_loss": float(best_val_loss),
                        "max_length": max_length,
                        "model_name": MODEL_NAME,
                        "use_4bit": USE_4BIT,
                        "epoch_history": epoch_history,
                    },
                )

        if best_adapter_state is None:
            raise RuntimeError(f"No best adapter state captured for fold {fold_idx}.")

        set_peft_model_state_dict(model, best_adapter_state)

        print(f"Predicting test set with best fold {fold_idx} adapter from epoch {best_epoch}...")
        test_probs_fold = predict_with_swap_tta(model, test_loader_orig, test_loader_swap, device)
        test_probs_accum += test_probs_fold.astype(np.float32)

        print(f"Best fold {fold_idx} validation log loss: {best_val_loss:.5f}")
        print(f"Fold {fold_idx} runtime: {format_seconds(time.time() - fold_start)}")
        fold_record = {
            "fold": fold_idx,
            "best_epoch": best_epoch,
            "best_val_log_loss": float(best_val_loss),
            "train_rows": int(len(train_idx)),
            "validation_rows": int(len(val_idx)),
            "runtime_seconds": int(time.time() - fold_start),
            "adapter_dir": str(OUTPUT_DIR / f"fold_{fold_idx}"),
            "epoch_history": epoch_history,
        }
        fold_metrics.append(fold_record)
        (OUTPUT_DIR / f"fold_{fold_idx}" / "fold_metrics.json").write_text(
            json.dumps(fold_record, indent=2),
            encoding="utf-8",
        )
        (OUTPUT_DIR / "fold_metrics.json").write_text(json.dumps(fold_metrics, indent=2), encoding="utf-8")

        del model, optimizer, scheduler, train_loader, val_loader_orig, val_loader_swap
        del test_loader_orig, test_loader_swap
        clear_torch_memory()

    oof_probs = normalize_probabilities(oof_probs)
    overall_log_loss = log_loss(labels, oof_probs, labels=[0, 1, 2])
    print_rule("Cross-validation summary")
    print(f"OOF log loss across {N_FOLDS} folds: {overall_log_loss:.5f}")

    test_probs = normalize_probabilities(test_probs_accum / N_FOLDS)
    submission_df = pd.DataFrame(
        {
            "id": test_df["id"],
            LABEL_COLUMNS[0]: test_probs[:, 0],
            LABEL_COLUMNS[1]: test_probs[:, 1],
            LABEL_COLUMNS[2]: test_probs[:, 2],
        }
    )
    validate_submission(submission_df, expected_rows=len(test_df))

    submission_path = OUTPUT_DIR / "submission.csv"
    submission_df.to_csv(submission_path, index=False)

    oof_path = OUTPUT_DIR / "oof_predictions.csv"
    pd.DataFrame(
        {
            "id": train_df["id"],
            LABEL_COLUMNS[0]: oof_probs[:, 0],
            LABEL_COLUMNS[1]: oof_probs[:, 1],
            LABEL_COLUMNS[2]: oof_probs[:, 2],
        }
    ).to_csv(oof_path, index=False)

    metrics = {
        "oof_log_loss": float(overall_log_loss),
        "fold_metrics": fold_metrics,
        "n_folds": N_FOLDS,
        "epochs": NUM_EPOCHS,
        "max_length_requested": MAX_LENGTH_REQUESTED,
        "max_length_effective": max_length,
        "train_rows": int(len(train_df)),
        "test_rows": int(len(test_df)),
        "model_name": MODEL_NAME,
        "adapter_method": "qlora" if USE_4BIT else "lora",
        "quantisation": f"4bit-{BNB_QUANT_TYPE}" if USE_4BIT else "none",
        "lora_rank": LORA_RANK,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "trainable_param_dtype": TRAINABLE_PARAM_DTYPE,
        "learning_rate": LEARNING_RATE,
        "label_smoothing": LABEL_SMOOTHING,
        "runtime_seconds": int(time.time() - run_start),
    }
    (OUTPUT_DIR / "run_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")

    print_rule("Submission file written")
    print(f"Submission path: {submission_path}")
    print(f"OOF path:        {oof_path}")
    print(f"Metrics path:    {OUTPUT_DIR / 'run_metrics.json'}")
    print(submission_df.head())
    print(f"Total runtime:   {format_seconds(time.time() - run_start)}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
